# 📘 DATABASE MANAGEMENT AND DESIGN
### *A Practical Learning Series*

# Notebook 3.3 — SQL and Data Analysis — Advanced Level I
### Joining Tables: Inner & Outer Joins

**By Amin Amirkhalili** — Business & Data Analyst

*Based on Chapter 2 (Section 2-3) and Chapter 6 (Sections 6-4 & 6-9) of:* **Database Management and Design** — Gove Allen, PhD · Gary Hansen, PhD · Robert Jackson, PhD


## 🗺️ Series Roadmap

This notebook is part of a practical learning series on database management and design. The series follows a logical progression — from understanding business needs, through conceptual modelling, all the way to writing SQL code on real databases.

| | |
|---|---|
| **1 — Fundamentals** | |
| Notebook 1 | Introduction: The Database Is the Heart of Information Systems |
| **2 — Database Management** | |
| Notebook 2.1 | Conceptual Data Model: From Conceptual Design to a Relational Model |
| Notebook 2.2 | Database Design: From Schema Creation to Data Management |
| **3 — SQL & Data Analysis** | |
| Notebook 3.1 | Beginner Level: SELECT, FROM, WHERE, ORDER BY |
| Notebook 3.2 | Intermediate Level: Functions, GROUP BY, HAVING |
| **Notebook 3.3** | Advanced Level I: Joining Tables — Inner & Outer Joins **◀ You are here** |
| Notebook 3.4 | Advanced Level II: Subqueries, Views, Temp Tables, CTEs |
| Notebook 3.5 | Data Cleaning with SQL *(coming soon)* |
| **4 — Data Mining** | |
| Notebook 4.1 | SQL in Data Mining *(coming soon)* |

## 📌 What This Notebook Covers

Why we join tables, inner joins (FK–PK and non-FK–PK), joining a table to itself, joining multiple tables, the 1987-style join, outer joins (left, right, full), and pitfalls when averaging over joined tables.

## 💻 How to Use This Notebook

This is the **live practice version** — every query below runs for real.

1. Run the setup cell below **first** (▶ button on the left). It builds a small sample database with the same tables used in the series.
2. Run each query cell as you read. Change the queries — experiment!
3. To start fresh, just run the setup cell again.

> **Note on SQL dialects:** the reading copy of this notebook uses **SQL Server (SSMS)**. Here we run **SQLite** (built into Colab, no installation needed). The two are almost identical; wherever the syntax differs (e.g. `TOP` vs `LIMIT`, `+` vs `||` for text), a note shows both versions.

In [ ]:
#@title ▶️ Run this cell first — it builds the practice database { display-mode: "form" }
# This cell creates a small sample database (SQLite) with the same tables used
# in the notebook: Customer, Manufacturer, Product, Sale, SaleItem, Employee,
# SalaryEmployee, WageEmployee, Purchase, Amin, SimplifiedSales, and more.
# It also defines q("...") which runs a SQL query and shows the result.

import sqlite3, math, datetime
import pandas as pd

conn = sqlite3.connect(":memory:")
conn.executescript("""
PRAGMA foreign_keys = ON;

CREATE TABLE Customer (
  CustomerID INT NOT NULL, FirstName VARCHAR(20), LastName VARCHAR(50),
  StreetAddress VARCHAR(80), City VARCHAR(40), State VARCHAR(2),
  PostalCode VARCHAR(10), Country VARCHAR(30), Phone VARCHAR(20),
  PRIMARY KEY (CustomerID));

CREATE TABLE Manufacturer (
  ManufacturerID INT NOT NULL, ManufacturerName VARCHAR(50),
  Address VARCHAR(80), City VARCHAR(40), State VARCHAR(2),
  PostalCode VARCHAR(10), Phone VARCHAR(20),
  PRIMARY KEY (ManufacturerID));

CREATE TABLE Product (
  ProductID INT NOT NULL, ProductName VARCHAR(60), ManufacturerID INT,
  Category VARCHAR(30), Color VARCHAR(20), Gender VARCHAR(1),
  ListPrice NUMERIC(8,2), Description VARCHAR(120),
  PRIMARY KEY (ProductID),
  FOREIGN KEY (ManufacturerID) REFERENCES Manufacturer(ManufacturerID));

CREATE TABLE Sale (
  SaleID INT NOT NULL, SaleDate DATE NOT NULL,
  Tax NUMERIC(8,2), Shipping NUMERIC(8,2), CustomerID INT,
  PRIMARY KEY (SaleID),
  FOREIGN KEY (CustomerID) REFERENCES Customer(CustomerID));

CREATE TABLE SaleItem (
  SaleID INT NOT NULL, ProductID INT NOT NULL, ItemSize NUMERIC(3,1) NOT NULL,
  Quantity INT NOT NULL, SalePrice NUMERIC(8,2) NOT NULL,
  PRIMARY KEY (SaleID, ProductID, ItemSize),
  FOREIGN KEY (SaleID) REFERENCES Sale(SaleID),
  FOREIGN KEY (ProductID) REFERENCES Product(ProductID));

CREATE TABLE Employee (
  EmployeeID INT NOT NULL, FirstName VARCHAR(20), LastName VARCHAR(50),
  Age INT, City VARCHAR(40), ManagerID INT,
  PRIMARY KEY (EmployeeID),
  FOREIGN KEY (ManagerID) REFERENCES Employee(EmployeeID));

CREATE TABLE SalaryEmployee (
  EmployeeID INT NOT NULL, Salary NUMERIC(10,2),
  PRIMARY KEY (EmployeeID),
  FOREIGN KEY (EmployeeID) REFERENCES Employee(EmployeeID));

CREATE TABLE WageEmployee (
  EmployeeID INT NOT NULL, Wage NUMERIC(8,2), MaxHours INT,
  PRIMARY KEY (EmployeeID),
  FOREIGN KEY (EmployeeID) REFERENCES Employee(EmployeeID));

CREATE TABLE Purchase (
  PurchaseID INT NOT NULL, PurchaseDate DATE, EmployeeID INT, ManufacturerID INT,
  PRIMARY KEY (PurchaseID),
  FOREIGN KEY (EmployeeID) REFERENCES Employee(EmployeeID),
  FOREIGN KEY (ManufacturerID) REFERENCES Manufacturer(ManufacturerID));

CREATE TABLE Amin (
  ID INT NOT NULL, FirstName VARCHAR(20), LastName VARCHAR(50),
  Age INT, Gender VARCHAR(10), PRIMARY KEY (ID));

CREATE TABLE Population (
  Country VARCHAR(40), City VARCHAR(40), Population INT);

CREATE TABLE CovidDeaths (
  location VARCHAR(40), continent VARCHAR(20), date DATE,
  total_cases INT, population INT);

INSERT INTO Manufacturer VALUES
 (1,'Nike','1 Bowerman Dr','Boston','NJ','07101','555-0101'),
 (2,'Adidas','5 Sport Ave','Boston','NJ','07102','555-0102'),
 (3,'Puma','9 Cat St','Chicago','IL','60601','555-0103'),
 (4,'Fila','3 Retro Rd','San Diego','CA','92101','555-0104'),
 (5,'Fashion Lab','7 Style Blvd','Los Angeles','CA','90001','555-0105'),
 (6,'GAP','11 Cotton Way','Concord','NH','03301','555-0106');

INSERT INTO Customer VALUES
 (1,'Sara','Lopez','12 Main St','Boston','VA','23220','USA','555-1001'),
 (2,'Amin','Amirkhalili','34 Oak Ave','Boston','MA','02108','USA','555-1002'),
 (3,'Kim','Nguyen','56 Pine Rd','Chicago','IL','60614','USA','555-1003'),
 (4,'John','Smith','78 Elm St','Richmond','VA','23230','USA','555-1004'),
 (5,'George','Brown','90 Cedar Ln','Springfield','IL','62701','USA','555-1005'),
 (6,'Kevin','Miller','21 Birch Ct','San Diego','CA','92103','USA','555-1006'),
 (7,'Sam','Wilson','43 Maple Dr','Portland','OR','97201','USA','555-1007'),
 (8,'Elli','Rahimi','65 Walnut St','Los Angeles','CA','90012','USA','555-1008');

INSERT INTO Product VALUES
 (1,'AirRun Sneaker',1,'Sneakers','Black','M',120,'Running sneaker'),
 (2,'AirRun Sneaker W',1,'Sneakers','White','F',115,'Running sneaker'),
 (3,'ClassicSandal',1,'Sandals','Brown','F',60,'Summer sandal'),
 (4,'StreetBoot',2,'Boots','Black','M',150,'Urban boot'),
 (5,'TrailBoot',2,'Boots','Brown','F',160,'Hiking boot'),
 (6,'SpeedCat Sneaker',3,'Sneakers','Red','M',95,'Classic sneaker'),
 (7,'BeachSandal',3,'Sandals','Gold','F',45,'Beach sandal'),
 (8,'RetroRunner',4,'Sneakers','Black','F',85,'Retro sneaker'),
 (9,'HeritageBoot',4,'Boots','Tan','M',140,'Heritage boot'),
 (10,'GlamHeel',5,'Heels','Black','F',130,'Evening heel'),
 (11,'CityFlat',5,'Flats','Blue','F',70,'Comfort flat'),
 (12,'CanvasClassic',6,'Sneakers','White','M',55,'Canvas sneaker'),
 (13,'WinterBoot',6,'Boots','Black','F',110,'Insulated boot');

INSERT INTO Sale VALUES
 (1001,'2026-07-03',8.40,5.00,1),
 (1002,'2026-07-02',6.20,5.00,2),
 (1003,'2026-06-28',12.10,0.00,3),
 (1004,'2026-06-15',4.75,5.00,4),
 (1005,'2026-05-30',9.90,7.50,2),
 (1006,'2026-01-18',7.25,5.00,5),
 (1007,'2026-01-05',3.60,0.00,6),
 (1008,'2025-12-20',10.10,5.00,1),
 (1009,'2025-12-08',5.45,5.00,8),
 (1010,'2026-07-10',6.80,0.00,3);

INSERT INTO SaleItem VALUES
 (1001,1,10.0,10,100.00),(1001,4,9.5,5,200.00),
 (1002,6,8.0,1,150.00),
 (1003,2,7.5,2,110.00),(1003,7,6.0,1,42.00),
 (1004,8,7.0,1,80.00),
 (1005,10,6.5,1,125.00),(1005,11,6.0,2,65.00),
 (1006,12,11.0,3,50.00),(1006,1,10.5,1,110.00),
 (1007,9,10.5,1,135.00),
 (1008,13,8.5,2,105.00),(1008,3,7.0,1,55.00),
 (1009,5,8.0,1,155.00),
 (1010,1,9.0,1,118.00);

INSERT INTO Employee VALUES
 (1,'Alan','Stone',55,'Boston',NULL),
 (2,'Ahmad','Karimi',41,'Boston',1),
 (3,'Elli','Moore',38,'Chicago',1),
 (4,'Luke','Perry',29,'San Diego',2),
 (5,'James','Chen',33,'San Diego',2),
 (6,'Alex','Diaz',26,'Portland',3);

INSERT INTO SalaryEmployee VALUES
 (1,95000),(2,72000),(3,68000),(4,80000);

INSERT INTO WageEmployee VALUES
 (5,28.50,40),(6,22.00,30);

INSERT INTO Purchase VALUES
 (501,'2025-12-05',2,1),
 (502,'2025-12-12',4,2),
 (503,'2025-12-27',5,3),
 (504,'2026-02-10',2,4),
 (505,'2026-03-22',3,5),
 (506,'2025-12-30',NULL,6);

INSERT INTO Amin VALUES
 (1,'Amin','Amirkhalili',30,'Male'),
 (2,'Amin','Amirkhalili',50,'Male'),
 (3,'Sara','Lopez',27,'Female'),
 (4,'Elli','Rahimi',35,'Female'),
 (5,'Sadra','Kazemi',22,'Male'),
 (6,'Hossein','Nouri',44,'Male'),
 (7,'Kim','Nguyen',31,'Female');

INSERT INTO Population VALUES
 ('Canada','Toronto',2794356),('Canada','Montreal',1762949),
 ('Canada','Vancouver',662248),('USA','New York',8804190),
 ('USA','Los Angeles',3898747),('USA','Chicago',2746388),
 ('Germany','Berlin',3644826),('Germany','Munich',1471508);

INSERT INTO CovidDeaths VALUES
 ('Canada','North America','2021-06-01',1385000,38000000),
 ('Canada','North America','2021-06-15',1401000,38000000),
 ('Canada','North America','2021-07-01',1412000,38000000),
 ('Germany','Europe','2021-06-01',3700000,83000000),
 ('Germany','Europe','2021-06-15',3717000,83000000),
 ('Germany','Europe','2021-07-01',3730000,83000000),
 ('Iran','Asia','2021-06-01',2950000,85000000),
 ('Iran','Asia','2021-06-15',3020000,85000000),
 ('Iran','Asia','2021-07-01',3230000,85000000);

CREATE TABLE SimplifiedSales AS
SELECT s.SaleID, s.SaleDate, s.CustomerID, p.ProductName, p.Category, p.Color,
       m.ManufacturerName, si.Quantity, si.SalePrice
FROM Sale s
JOIN SaleItem si ON s.SaleID = si.SaleID
JOIN Product p  ON si.ProductID = p.ProductID
JOIN Manufacturer m ON p.ManufacturerID = m.ManufacturerID;
""")

# --- Register SQL-Server-style functions so the book's code runs in SQLite ---
conn.create_function("LEN", 1, lambda s: len(s) if s is not None else None)
conn.create_function("CHARINDEX", 2, lambda sub, s: (s.find(sub) + 1) if (s is not None and sub is not None) else None)
conn.create_function("SQRT", 1, lambda x: math.sqrt(x) if x is not None else None)
conn.create_function("POWER", 2, lambda x, n: x ** n if x is not None else None)
conn.create_function("CEILING", 1, lambda x: math.ceil(x) if x is not None else None)
conn.create_function("FLOOR", 1, lambda x: math.floor(x) if x is not None else None)
conn.create_function("GETDATE", 0, lambda: datetime.date.today().isoformat())
conn.create_function("MONTH", 1, lambda d: int(str(d)[5:7]) if d else None)
conn.create_function("YEAR", 1, lambda d: int(str(d)[0:4]) if d else None)

class _StringAgg:
    def __init__(self): self.items = []
    def step(self, value, sep): self.sep = sep; self.items.append(str(value))
    def finalize(self): return self.sep.join(self.items) if self.items else None
conn.create_aggregate("STRING_AGG", 2, _StringAgg)

def q(sql):
    """Run a SQL query and show the result as a table."""
    sql_stripped = sql.strip().rstrip(";").strip()
    first = sql_stripped.split()[0].upper() if sql_stripped else ""
    if first in ("SELECT", "WITH"):
        return pd.read_sql_query(sql, conn)
    conn.executescript(sql)
    print("OK —", first, "executed.")

print("✅ Practice database is ready. Use q(\"...\") to run SQL. Example:")
q("SELECT * FROM Customer LIMIT 3")


## Why Do We Need to Join Tables?

Before answering that, we need to answer a fundamental question: does all data have to be in one table? (If so, we would not need to join tables.)

**The answer is no. But why?**

Because keeping everything in one table causes:

1. **Unwanted repetition in data entry.** Suppose we record a location for each sale made in a chain store. Then the location has to be entered every time. It is wiser to keep all locations in a different table, and in the Sale table only keep related and more dynamic info.
2. **Inconvenience when some data has been corrupted or needs revision.** If the location of one store has been recorded incorrectly, all the rows need to be revised. But if locations live in a different table, we only need to change it once.

**Therefore, the solution is to separate each independent entity into separate tables (normalization).**

Now, because we have separate tables in our database, we need to **join** them to answer some of the queries.

### Types of joins
- Inner join
- Outer join: left outer, right outer, full outer

## Inner Join

INNER JOIN is a synonym for the common term JOIN. It joins two tables together but only includes rows that meet the matching criteria. It means: only show the rows that have a complete match between the two (or more) tables. This is the most common join used in SQL.

### 1. Joining two tables together

We can join two tables with:
- **FK–PK** (foreign key to primary key)
- **Non FK–PK**

The `JOIN ... ON` version first became a standard in 1992. In a prior version of SQL (the 1987 standard), the join condition was placed within the WHERE clause — we show that later.

#### FK–PK
The FK is usually in the child table, and the primary key is in the father table.

```sql
SELECT * FROM Left_Table
JOIN Right_Table ON ForeignKey = PrimaryKey
WHERE ...
```

In [ ]:
q("""
SELECT * FROM Product P
JOIN Manufacturer M ON P.ManufacturerID = M.ManufacturerID
WHERE City = 'Boston'
""")

In joined tables, all attributes from both tables are shown in the result. But if we write `TableName.*` in SELECT, only that table's attributes are shown. To avoid writing long names, we use an **alias**:

In [ ]:
q("""
SELECT P.*
FROM Product P
JOIN Manufacturer M
  ON P.ManufacturerID = M.ManufacturerID
""")

If we want all attributes from one table and some attributes from the other:

In [ ]:
q("""
SELECT P.*, M.ManufacturerName
FROM Product P
JOIN Manufacturer M
  ON P.ManufacturerID = M.ManufacturerID
""")

As mentioned before, JOIN comes after SELECT and FROM, before anything else:

In [ ]:
q("""
SELECT ProductID, ProductName, ListPrice, Gender, Category, ManufacturerName
FROM Product P
JOIN Manufacturer M
  ON P.ManufacturerID = M.ManufacturerID
WHERE ManufacturerName = 'Fila'
""")

#### Could we join any two attributes in two tables?

**No.** The join condition fields must be the same data type and have the same range of values. Joining `P.ManufacturerID = M.City` would be meaningless.

#### Non FK–PK

We can join any non foreign-key–primary-key pair together — but they must have the same data type and range.

**Query: list the names of customers that live in the same city as a shoe manufacturer. Include the manufacturer name and the city name.**

In [ ]:
q("""
SELECT
    CustomerID,
    FirstName,
    LastName,
    ManufacturerName,
    m.City,
    c.City
FROM Manufacturer m
JOIN Customer c ON c.City = m.City
""")

### A limitation of join

One of the main limitations of join is that it produces repetition, which is redundancy: if Sara and Amin live in Boston and both Nike and Adidas are in Boston, the join gives four rows. It is not wrong (this form is normal). But one other way to show it — not normalized, yet easy to understand — is one row per customer. How can we make it? With GROUP BY and `STRING_AGG`:

In [ ]:
q("""
SELECT
    c.CustomerID,
    c.FirstName,
    c.LastName,
    c.City,
    STRING_AGG(m.ManufacturerName, ', ') AS Manufacturers
FROM Customer c
JOIN Manufacturer m ON c.City = m.City
GROUP BY
    c.CustomerID,
    c.FirstName,
    c.LastName,
    c.City
ORDER BY c.City
""")

Therefore, we only have unique customers (customers do not repeat, but their cities repeat). BUT, this form is not normalized.

*(SQLite's native name for STRING_AGG is `GROUP_CONCAT` — both work here.)*

### 2. Joining a table to itself

One of the main uses of joining a table to itself is to show organizational hierarchy:

In [ ]:
q("""
SELECT E.FirstName AS EmpFirstName,
       E.LastName  AS EmpLastName,
       M.FirstName AS MgrFirstName,
       M.LastName  AS MgrLastName
FROM Employee E
JOIN Employee M
  ON E.ManagerID = M.EmployeeID
""")

### 3. Joining multiple tables

**Joining 3 tables — query: list products made in New Jersey that have sold in January.**

*(In our sample schema, Sale connects to Product through SaleItem, so the “3 tables” here are Sale → SaleItem → Product → Manufacturer.)*

In [ ]:
q("""
SELECT DISTINCT ProductName
FROM Sale S
JOIN SaleItem SI
  ON S.SaleID = SI.SaleID
JOIN Product P
  ON SI.ProductID = P.ProductID
JOIN Manufacturer M
  ON P.ManufacturerID = M.ManufacturerID
WHERE State = 'NJ'
  AND MONTH(SaleDate) = 1
""")

**Query: list sales of black sneakers.**

In [ ]:
q("""
SELECT S.SaleDate,
       P.ProductName,
       P.Category,
       P.Color,
       SI.Quantity,
       SI.SalePrice
FROM Sale S
JOIN SaleItem SI
  ON S.SaleID = SI.SaleID
JOIN Product P
  ON SI.ProductID = P.ProductID
WHERE Category = 'Sneakers'
  AND Color = 'Black'
""")

**Joining four tables — query: list products made in New Jersey sold to customers in Illinois.**

In [ ]:
q("""
SELECT DISTINCT ProductName
FROM Sale S
JOIN SaleItem SI
  ON S.SaleID = SI.SaleID
JOIN Product P
  ON P.ProductID = SI.ProductID
JOIN Customer C
  ON S.CustomerID = C.CustomerID
JOIN Manufacturer M
  ON P.ManufacturerID = M.ManufacturerID
WHERE C.State = 'IL'
  AND M.State = 'NJ'
""")

**The final example — query: list the customers who have made at least one purchase of a product from a manufacturer located in the same city as the customer. Return the customer's first and last name, the manufacturer name, and the shared city (one row per matching customer–manufacturer pair), ordered by city.**

In [ ]:
q("""
SELECT DISTINCT
    c.FirstName,
    c.LastName,
    m.ManufacturerName,
    c.City
FROM Customer c
JOIN Sale s      ON s.CustomerID = c.CustomerID
JOIN SaleItem si ON si.SaleID = s.SaleID
JOIN Product p   ON p.ProductID = si.ProductID
JOIN Manufacturer m ON m.ManufacturerID = p.ManufacturerID
WHERE c.City = m.City
ORDER BY c.City
""")

**In general, it's easy to see that if we join n tables, we'll have n−1 join conditions.** Therefore, we can join as many tables as we need in order to answer a given query.

### Join, 1987 style (in the WHERE clause)

- The tables to be joined are simply listed in the FROM clause, separated by commas.
- The joining conditions (i.e. the ON clause) are listed in the WHERE clause.

**Query: list sales details for black sneakers.**

In [ ]:
q("""
SELECT S.SaleDate,
       P.ProductName,
       P.Category,
       P.Color,
       SI.Quantity,
       SI.SalePrice
FROM Sale S, SaleItem SI, Product P
WHERE S.SaleID = SI.SaleID
  AND SI.ProductID = P.ProductID
  AND Category = 'Sneakers'
  AND Color = 'Black'
""")

## Outer Join

An OUTER JOIN returns all rows from one (or both) of the tables, even if the matching criteria fails.

> ### 💡 How do I know which OUTER JOIN to use?
> - **LEFT OUTER JOIN**: all rows from the left table, including unmatched rows. (The left table is the first one listed.)
> - **RIGHT OUTER JOIN**: all rows from the right table, including unmatched rows. (The right table is the second one listed.)
> - **FULL OUTER JOIN**: all rows of both tables, even unmatched rows.

Suppose we have employee information and their purchases in two tables. First, let's create them (these are views — we learn views properly in Notebook 3.4):

In [ ]:
q("""
DROP VIEW IF EXISTS SalEmployee;
CREATE VIEW SalEmployee AS
SELECT E.EmployeeID, E.FirstName, E.LastName, E.City, SE.Salary
FROM Employee E JOIN SalaryEmployee SE ON E.EmployeeID = SE.EmployeeID;

DROP VIEW IF EXISTS DecPurchase;
CREATE VIEW DecPurchase AS
SELECT * FROM Purchase WHERE MONTH(PurchaseDate) = 12;
""")

### Example — left outer join

**Query: list all salaried employees (EmployeeID, FirstName, LastName) along with the date and manufacturer ID of their December purchases. Include salaried employees that have made no purchases.**

Left table: SalEmployee. Right table: DecPurchase.

In [ ]:
q("""
SELECT E.EmployeeID, FirstName, LastName, PurchaseDate, ManufacturerID
FROM SalEmployee E LEFT OUTER JOIN DecPurchase P
  ON E.EmployeeID = P.EmployeeID
""")

It means: show all employees, whether they have purchased anything or not.

### Example — right outer join

**Query: list all the December purchases (PurchaseDate, ManufacturerID) and the associated salaried employees. Include all December purchases.**

*(RIGHT and FULL OUTER JOIN need SQLite 3.39+ — Google Colab has it. If you ever hit an error on an old SQLite, swap the table order and use LEFT JOIN.)*

In [ ]:
q("""
SELECT E.EmployeeID, FirstName, LastName, PurchaseDate, ManufacturerID
FROM SalEmployee E RIGHT OUTER JOIN DecPurchase P
  ON E.EmployeeID = P.EmployeeID
""")

It means: show all the purchases, whether they are connected to any employee or not.

### Example — full outer join

**Query: list all salaried employees and all December purchases, showing which December purchases were made by salaried employees.**

In [ ]:
q("""
SELECT E.EmployeeID, FirstName, LastName, PurchaseDate, ManufacturerID
FROM SalEmployee E FULL OUTER JOIN DecPurchase P
  ON E.EmployeeID = P.EmployeeID
""")

It means: show everything — the employees who purchased (inner join), the employees who have not purchased (left outer join), and the purchases that are not related to any employee (right outer join).

### WHERE with outer joins — be careful

Attention: we can still use a WHERE clause with an outer join, but we have to make sure it does not turn the outer join into an inner join. Here the left outer join performs correctly, because the WHERE clause applies to the left table:

In [ ]:
q("""
SELECT
    E.EmployeeID, E.FirstName, E.LastName, E.City,
    P.PurchaseDate, P.ManufacturerID
FROM
    SalEmployee E LEFT OUTER JOIN DecPurchase P
    ON E.EmployeeID = P.EmployeeID
WHERE E.City = 'San Diego'
""")

Now, suppose we want to see **all** the employees who purchased something in month 12 **or have not purchased anything**. If we write this query:

In [ ]:
q("""
SELECT E.EmployeeID, FirstName, LastName, PurchaseDate, ManufacturerID
FROM SalEmployee E LEFT OUTER JOIN Purchase P
  ON E.EmployeeID = P.EmployeeID
WHERE MONTH(PurchaseDate) = 12
""")

First FROM applies (the join is done). Second, WHERE applies — only the ones with a purchase in month 12 are picked, so all rows without purchase data are dropped (but we wanted them). Third, SELECT applies. It means the left outer join no longer has any effect: we don't see the employees who have not purchased anything.

**What is the solution?** There are two solutions:

1. Using a view (Notebook 3.4).
2. Using the condition inside the JOIN statement:

In [ ]:
q("""
SELECT E.EmployeeID, FirstName, LastName, PurchaseDate, ManufacturerID
FROM SalEmployee E
LEFT OUTER JOIN Purchase P
  ON E.EmployeeID = P.EmployeeID
 AND MONTH(P.PurchaseDate) = 12
""")

Here, first FROM applies (the join), and at the same time the month-12 check happens: if a match exists, use the data; if not, use NULL. Then SELECT applies. Therefore, the left outer join works properly.

## Avoiding the Pitfalls in Join

**Query: what is the average list price of products sold this month?**

If we only wanted the average list price of products [not in any month], we would not need any joining:

In [ ]:
q("""
SELECT AVG(ListPrice) FROM Product
""")

BUT the Product table does not have any time attribute. So we join — and here is the trap. A product sold in quantity 10 gets counted 10 times:

In [ ]:
q("""
SELECT AVG(ListPrice)
FROM Sale S
JOIN SaleItem SI ON S.SaleID = SI.SaleID
JOIN Product P  ON SI.ProductID = P.ProductID
WHERE MONTH(SaleDate) = MONTH(GETDATE())
""")

What if we calculate the average based on sale price instead of list price — a **weighted** average?

In [ ]:
q("""
SELECT (SUM(SalePrice * Quantity) / SUM(Quantity)) AS WeightedAverageSalePrice
FROM Sale S
JOIN SaleItem SI ON S.SaleID = SI.SaleID
JOIN Product P  ON SI.ProductID = P.ProductID
WHERE MONTH(SaleDate) = MONTH(GETDATE())
""")

Another interpretation: count each product **once** if it was sold in the month — using subqueries (a preview of Notebook 3.4):

In [ ]:
q("""
SELECT AVG(ListPrice) AS AverageListPrice
FROM Product
WHERE ProductID IN
    (SELECT ProductID
     FROM SaleItem
     WHERE SaleID IN
         (SELECT SaleID
          FROM Sale
          WHERE MONTH(SaleDate) = MONTH(GETDATE())
         )
    )
""")

---
### 🎯 Your turn — practice ideas

1. List every product with its manufacturer name and city.
2. Which customers have never made a purchase? (LEFT JOIN + `WHERE ... IS NULL`)
3. For each manufacturer, how many distinct products have been sold?

---
⏭️ **Coming up in Notebook 3.4:** subqueries, views, temp tables, and CTEs.